# Experiment 3 — XGBoost + SMOTE

Creates synthetic signal events by interpolating between existing ones.

**CRITICAL RULES:**
- SMOTE applied ONLY to X_train — NEVER to test set (data leakage)
- Version A cap: 500k rows (CPU/RAM constraint) — noted as paper limitation
- Versions B and C: NO cap — use full training set (already small due to downsampling)

**Prerequisite:** Run `00_dataset_prep.ipynb` first.

In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

sys.path.insert(0, os.path.abspath('..'))
from utils.metrics import compute_all_metrics, print_metrics

DATA_DIR    = os.path.join('..', 'data')
RESULTS_DIR = os.path.join('..', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# FIXED: removed use_label_encoder=False (removed in XGBoost 2.0+, causes TypeError)
# ADDED: tree_method='hist' — much faster on large datasets (11M rows)
# ADDED: n_jobs=-1 — use all CPU cores
# ADDED: device='cpu' — explicit, avoids XGBoost 2.x warnings
PARAMS = dict(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='auc',
    verbosity=0,
    tree_method='hist',
    device='cpu',
    n_jobs=-1
)

# SMOTE CAPS — Version A has huge training set on full 11M run,
# so we cap it to prevent RAM exhaustion (16GB limit).
# B and C are already small due to downsampling — no cap needed.
SMOTE_CAP_VERSION_A = 500_000

print(f"Experiment 3 — XGBoost + SMOTE | Version A cap: {SMOTE_CAP_VERSION_A:,} rows")
print("Params:", PARAMS)

In [ ]:
all_results = []

for v in ['A', 'B', 'C']:
    print(f"\n[Exp3-XGB] Processing Version {v}...")
    try:
        train = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_train.csv'))
        test  = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_test.csv'))
    except FileNotFoundError:
        print(f"ERROR: Run 00_dataset_prep.ipynb first."); raise

    X_train = train.drop('label', axis=1).values
    y_train = train['label'].values
    X_test  = test.drop('label', axis=1).values
    y_test  = test['label'].values

    # Free the dataframes immediately — SMOTE will create large arrays,
    # so we need every MB we can free on 16GB RAM with 11M rows
    del train, test

    # Cap applies ONLY to Version A — B and C use full training data
    if v == 'A' and len(X_train) > SMOTE_CAP_VERSION_A:
        print(f"[Exp3-XGB] Version A: capping to {SMOTE_CAP_VERSION_A:,} rows for SMOTE.")
        idx     = np.random.RandomState(42).choice(len(X_train), SMOTE_CAP_VERSION_A, replace=False)
        X_smote = X_train[idx]
        y_smote = y_train[idx]
        # Free full arrays — we only need the capped subset now
        del X_train, y_train
    else:
        print(f"[Exp3-XGB] Version {v}: full training set ({len(X_train):,} rows) for SMOTE.")
        X_smote = X_train
        y_smote = y_train
        del X_train, y_train

    # SMOTE: n_jobs=-1 uses all cores — critical for large arrays
    smote   = SMOTE(random_state=42, n_jobs=-1)
    t_smote = time.time()
    X_res, y_res = smote.fit_resample(X_smote, y_smote)
    smote_time   = time.time() - t_smote

    # Free SMOTE input arrays — X_res/y_res already hold what we need
    del X_smote, y_smote

    print(f"[Exp3-XGB] SMOTE {smote_time:.1f}s | {len(X_res):,} rows | Sig={y_res.sum():,} | BG={(y_res==0).sum():,}")

    model = XGBClassifier(**PARAMS)
    t0    = time.time()
    model.fit(X_res, y_res)
    train_time = time.time() - t0 + smote_time

    # Free resampled arrays before predict — model is built, no longer needed
    del X_res, y_res

    probs = model.predict_proba(X_test)[:, 1]
    preds = model.predict(X_test)

    metrics = compute_all_metrics(y_test, probs, preds, train_time)
    metrics['Version'] = v
    all_results.append(metrics)

    np.save(os.path.join(RESULTS_DIR, f'probs_exp3_xgb_version_{v}.npy'), probs)
    print_metrics(metrics, label=f"Exp3-XGB Version {v}")

    # Free everything before next version
    del X_test, y_test, model, probs, preds

print("\n[Exp3-XGB] Complete.")

In [ ]:
results_df = pd.DataFrame(all_results)[['Version', 'AUC', 'F1', 'Signal_Significance', 'Train_Time_sec']]
save_path  = os.path.join(RESULTS_DIR, 'experiment3_smote_xgb.csv')
results_df.to_csv(save_path, index=False)
print(results_df.to_string(index=False))
print(f"\n✓ Saved → {save_path}")